# RSS Fine-mapping with GWAS Summary Statistics

## Description

This notebook performs high-dimensional regression with GWAS summary statistics using the **Regression with Summary Statistics (RSS)** model. Given a list of GWAS summary-statistic files and an LD-reference panel, it fine-maps causal variants region by region with the SuSiE-RSS model (and supports conditional-regression variants).

The workflow has three steps: `get_analysis_regions` builds the list of genomic regions to analyze (from the LD-reference metadata and/or user-supplied regions); `univariate_rss` runs allele-quality control, optional summary-statistic imputation (RAISS), and the chosen fine-mapping method on each region; `univariate_plot` renders a PIP plot for a result.

**Synthetic-data note:** the minimal working example below uses a small, clearly-labelled toy GWAS summary-statistic file (`protocol_example.gwas_sumstats.chr22.tsv.gz`) derived from a public Alzheimer's GWAS, restricted to a single chr22 region, together with the toy PLINK LD reference under `input/ld_reference/`. No access-controlled individual-level genotype data is used.

## LD reference and the LD sketch

RSS fine-mapping relies on a **linkage-disequilibrium (LD) reference** that describes the correlation structure among variants in each region. For large cohorts the full per-region LD matrices are big, so the protocol uses an **LD sketch**: a compact, region-partitioned representation of the LD reference (low-rank / block-sparse summaries plus per-region metadata) that is precomputed by the LD reference processing module. `get_analysis_regions` reads this LD-sketch metadata to enumerate the regions to fine-map, and `univariate_rss` loads the matching per-region LD block for allele QC and the SuSiE-RSS fit.

For this minimal working example the toy PLINK LD reference under `input/ld_reference/` already plays the role of the LD sketch for the single chr22 region analyzed here; on real data you would point `--ld-reference-meta` at the LD-sketch metadata produced for your cohort.

## Input

| File | Description |
|------|-------------|
| `--gwas-meta-data` | TSV listing each GWAS study: columns `study_id`, `chrom` (0 = genome-wide), `file_path` (summary-statistics file), `column_mapping_file` (YAML mapping standard column names), `n_sample`, `n_case`, `n_control`. Paths are resolved relative to this file's directory or the project root. |
| `--ld-meta-data` | TSV describing the LD reference: columns `#chrom`, `start`, `end`, `path`. Two formats are supported: pre-computed LD blocks (`.cor.xz` with `ld_file,bim_file`) or per-chromosome PLINK genotype prefixes (`.pgen/.pvar/.psam` or `.bed/.bim/.fam`) with `start=0`, `end=0`. |
| column-mapping YAML | Maps standard names (`chrom`, `pos`, `A1`, `A2`, `z` or `beta`+`se`, `pvalue`, `maf`, `n_case`, `n_control`, `n_sample`) to the original column names in the summary-statistics file. |
| `--region-name` / `--region-list` | One or more regions in `chr:start-end` format, or a file listing them. If neither is given, all regions in the LD-reference metadata are analyzed. |

Key parameters: `--qc-method` (`slalom` or `dentist`), `--impute` (impute summary stats for SNPs in LD but missing from sumstats via RAISS), `--finemapping-method` (`single_effect`, `susie_rss`, or `bayesian_conditional_regression`), `--L` (max number of causal effects), `--maf` / `--imiss` filtering thresholds, `--skip-analysis-pip-cutoff`.

## Output

All results are written under `--cwd`, one set of files per analyzed LD-block region:

- `{qc_method}_{imputed}.chr{N}_{start}_{end}.univariate_susie_rss.rds` — the SuSiE-RSS fine-mapping result for that block (e.g. `SLALOM_RAISS_imputed.chr22_49355984_50799822.univariate_susie_rss.rds`), holding the `single_effect_regression`, `noqc`, `qc_only`, `qc_impute` and (where applicable) `conditional_regression_*` objects plus the QC'd / imputed `sumstats_*` tables; an extra diagnostic table is added when `--diagnostics` is set.
- `{prefix}.sumstats.tsv.gz` — a companion harmonized summary-statistics table accompanying each RDS for convenience.
- `{step_name}/*.pip_plot.png`, plus per-region `.stderr` / `.stdout` logs — the posterior-inclusion-probability plot and run logs.


## Steps

**Step 1.** Build the list of analysis regions from the LD-reference metadata. This step is normally invoked automatically as a dependency of `univariate_rss`, but it can be run on its own to inspect the regions that will be analyzed.

**Timing:** ~3-5 min on typical compute infrastructure.

In [ ]:
sos run pipeline/rss_analysis.ipynb get_analysis_regions \
    --ld-meta-data input/ld_reference/protocol_example.ld_meta_file.tsv \
    --gwas-meta-data input/rss_analysis/protocol_example.gwas_meta_data.tsv \
    --region-name 22:49355984-50799822 \
    --cwd output/rss_analysis

**Step 2.** Fine-map the region with the RSS model. This runs allele QC (`--qc-method slalom`), imputes missing summary statistics against the LD reference (`--impute`, RAISS), and fits SuSiE-RSS (`--finemapping-method susie_rss`), writing one `.rds` result per region.

In [ ]:
sos run pipeline/rss_analysis.ipynb univariate_rss \
    --ld-meta-data input/ld_reference/protocol_example.ld_meta_file.tsv \
    --gwas-meta-data input/rss_analysis/protocol_example.gwas_meta_data.tsv \
    --qc-method slalom --impute \
    --finemapping-method susie_rss \
    --skip-analysis-pip-cutoff 0 \
    --region-name 22:49355984-50799822 \
    --cwd output/rss_analysis

**Step 3.** Render a PIP (posterior inclusion probability) plot from a fine-mapping result `.rds` produced in Step 2. Point `--region-list` / inputs at the result of interest; the plot is written as a PNG under `--cwd`.

In [ ]:
sos run pipeline/rss_analysis.ipynb univariate_plot \
    --ld-meta-data input/ld_reference/protocol_example.ld_meta_file.tsv \
    --gwas-meta-data input/rss_analysis/protocol_example.gwas_meta_data.tsv \
    --region-name 22:49355984-50799822 \
    --cwd output/rss_analysis \
    --input output/rss_analysis/univariate_rss/SLALOM_RAISS_imputed.chr22_49355984_50799822.univariate_susie_rss.rds

## Command interface

Run the cell below to list every workflow and its options.

In [ ]:
sos run pipeline/rss_analysis.ipynb -h

## Workflow implementation

The cells below contain the original SoS workflow definitions (`[global]`, `[get_analysis_regions]`, `[univariate_rss]`, `[univariate_plot]`) preserved verbatim.

In [ ]:
[global]
parameter: modular_script_dir = path('code/script')  # override with --modular-script-dir
parameter: cwd = path("output/")
parameter: gwas_meta_data = path()
parameter: ld_meta_data = path()
parameter: gwas_name = []
parameter: gwas_data = []
parameter: column_mapping = []
parameter: region_list = path()
parameter: region_name = []
parameter: ld_reference_size = 10000
parameter: ld_mismatch_correction = False
parameter: container = ''
parameter: skip_regions = []
import re
parameter: entrypoint= ('micromamba run -a "" -n' + ' ' + re.sub(r'(_apptainer:latest|_docker:latest|\.sif)$', '', container.split('/')[-1])) if container else ""
parameter: job_size = 5
parameter: walltime = "10h"
parameter: mem = "16G"
parameter: numThreads = 1
def group_by_region(lst, partition):
    # from itertools import accumulate
    # partition = [len(x) for x in partition]
    # Compute the cumulative sums once
    # cumsum_vector = list(accumulate(partition))
    # Use slicing based on the cumulative sums
    # return [lst[(cumsum_vector[i-1] if i > 0 else 0):cumsum_vector[i]] for i in range(len(partition))]
    return partition
import os
if (not os.path.isfile(region_list)) and len(region_name) == 0:
    region_list = ld_meta_data

In [ ]:
[get_analysis_regions: shared = "regional_data"]
import os
import pandas as pd
from collections import OrderedDict

def file_exists(file_path, relative_path=None):
    """Check if a file exists at the given path or relative to a specified path."""
    if os.path.exists(file_path) and os.path.isfile(file_path):
        return True
    elif relative_path:
        relative_file_path = os.path.join(relative_path, file_path)
        return os.path.exists(relative_file_path) and os.path.isfile(relative_file_path)
    return False

def check_required_columns(df, required_columns):
    """Check if the required columns are present in the dataframe."""
    missing_columns = [col for col in required_columns if col not in df.columns]
    if missing_columns:
        raise ValueError(f"Missing required columns: {', '.join(missing_columns)}")

def parse_region(region):
    """Parse a region string in 'chr:start-end' format into a list [chr, start, end]."""
    chrom, rest = region.split(':')
    start, end = rest.split('-')
    return [int(chrom), int(start), int(end)]

def load_regional_rss_data(gwas_meta_data, gwas_name, gwas_data, column_mapping, region_name=None, region_list=None):
    """
    Extracts data from GWAS metadata files and additional GWAS data provided. 
    Optionally filters data based on specified regions.

    Args:
    - gwas_meta_data (str): File path to the GWAS metadata file.
    - gwas_name (list): Vector of GWAS study names.
    - gwas_data (list): Vector of GWAS data.
    - column_mapping (list, optional): Vector of column mapping files.
    - region_name (list, optional): List of region names in 'chr:start-end' format.
    - region_list (str, optional): File path to a file containing regions.

    Returns:
    - GWAS Dictionary: Maps study IDs to a list containing chromosome number, 
      GWAS file path, and optional column mapping file path.
    - Region Dictionary: Maps region names to lists [chr, start, end].

    Raises:
    - FileNotFoundError: If any specified file path does not exist.
    - ValueError: If required columns are missing in the input files or vector lengths mismatch.
    """
    # Check vector lengths
    if len(gwas_name) != len(gwas_data):
        raise ValueError("gwas_name and gwas_data must be of equal length")
    
    if len(column_mapping) > 0 and len(column_mapping) != len(gwas_name):
        raise ValueError("If column_mapping is provided, it must be of the same length as gwas_name and gwas_data")

    # Required columns for GWAS file type
    required_gwas_columns = ['study_id', 'chrom', 'file_path']

    # Base directory of the metadata files
    gwas_base_dir = os.path.dirname(gwas_meta_data)
    
    # Reading the GWAS metadata file
    gwas_df = pd.read_csv(gwas_meta_data, sep="\t")
    check_required_columns(gwas_df, required_gwas_columns)
    gwas_dict = OrderedDict()

    # Process additional GWAS data from vectors
    for name, data, mapping in zip(gwas_name, gwas_data, column_mapping or [None]*len(gwas_name)):
        gwas_dict[name] = {0: [data, mapping]}

    for _, row in gwas_df.iterrows():
        file_path = row['file_path']
        mapping_file = row.get('column_mapping_file')
        n_sample = row.get('n_sample')
        n_case = row.get('n_case')
        n_control = row.get('n_control')

        # Check if the file and optional mapping file exist
        if not file_exists(file_path, gwas_base_dir) or (mapping_file and not file_exists(mapping_file, gwas_base_dir)):
            raise FileNotFoundError(f"File {file_path} not found for {row['study_id']}")
        
        # Adjust paths if necessary
        file_path = file_path if file_exists(file_path) else os.path.join(gwas_base_dir, file_path)
        if mapping_file:
            mapping_file = mapping_file if file_exists(mapping_file) else os.path.join(gwas_base_dir, mapping_file)
        
        # Create or update the entry for the study_id
        if row['study_id'] not in gwas_dict:
            gwas_dict[row['study_id']] = {}

        # Expand chrom 0 to chrom 1-22 or use the specified chrom
        chrom_range = range(1, 23) if row['chrom'] == 0 else [row['chrom']]
        for chrom in chrom_range:
            if chrom in gwas_dict[row['study_id']]:
                existing_entry = gwas_dict[row['study_id']][chrom]
                raise ValueError(f"Duplicate chromosome specification for study_id {row['study_id']}, chrom {chrom}. "
                                 f"Conflicting entries: {existing_entry} and {[file_path, mapping_file]}")
            gwas_dict[row['study_id']][chrom] = [file_path, mapping_file, n_sample, n_case, n_control, row.get('ld_meta_data', str(ld_meta_data))]

    # Process region_list and region_name
    region_dict = dict()
    if region_list and os.path.isfile(region_list):
        with open(region_list, 'r') as file:
            for line in file:
                # Skip empty lines
                if not line.strip():
                    continue
                if line.startswith("#"):
                    continue
                parts = line.strip().split()
                if len(parts) == 1:
                    region = parse_region(parts[0])
                elif len(parts) == 3:
                    region = [int(parts[0].replace("chr", "")), int(parts[1]), int(parts[2])]
                elif len(parts) >= 4 and  region_list != ld_meta_data : # for eQTL where chr:start:end:gene_id:gene_name, and path if LD_meta are used.
                    region = [int(parts[0].replace("chr", "")), int(parts[1]), int(parts[2]),parts[3]]
                else:
                    raise ValueError("Invalid region format in region_list")
        
                region_dict[f"{region[0]}:{region[1]}_{region[2]}"] = region

    if region_name:
        for region in region_name:
            parsed_region = parse_region(region)
            region_key = f"{parsed_region[0]}:{parsed_region[1]}_{parsed_region[2]}"
            if region_key not in region_dict:
                region_dict[region_key] = parsed_region

    return gwas_dict, region_dict

gwas_dict, region_dict = load_regional_rss_data(gwas_meta_data, gwas_name, gwas_data, column_mapping, region_name, region_list)
regional_data = dict([("GWAS", gwas_dict), ("regions", region_dict)])

In [ ]:
[univariate_rss]
parameter: L = 15
parameter: L_greedy = 5
# If available the column that indicates sample size within the sumstats
# filtering threshold for raiss imputation
parameter: rcond = 0.01
parameter: lamb = 0.01
parameter: R2_threshold = 0.6
parameter: pip_cutoff = 0.025
parameter: skip_analysis_pip_cutoff = 0.025
parameter: minimum_ld = 5
parameter: coverage = [0.95, 0.7, 0.5]
# Whether to impute the sumstat for all the snp in LD but not in sumstat.
parameter: impute = False 
# summary stats QC methods: slalom, dentist
parameter: qc_method = ""
# analysis method: single_effect, susie_rss, bayesian_conditional_regression
parameter: finemapping_method = "single_effect"
parameter: extract_region_name = "NULL"
parameter: region_name_col = "NULL"
# remove a variant if it has more than imiss missing individual level data
parameter: imiss = 1.0
# MAF cutoff
parameter: maf = 0.0025
parameter: comment_string = "NULL"
parameter: diagnostics = False
depends: sos_variable("regional_data")
regions = list(regional_data['regions'].keys())
input: for_each = "regions"
output: f'{cwd:a}/{step_name}/{"%s" % qc_method.upper() if qc_method else "noQC"}{"_RAISS_imputed" if impute else ""}.chr{_regions.replace(":", "_")}.univariate{"_%s" % finemapping_method if finemapping_method else ""}.rds' if not diagnostics else f'{cwd:a}/{step_name}/Automated_QC.chr{_regions.replace(":", "_")}.univariate.rds'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f'{step_name}_{_output:bn}'
bash: expand= "${ }", stderr = f"{_output:n}.stderr", stdout = f"{_output:n}.stdout", container = container, entrypoint = entrypoint
    Rscript ${modular_script_dir}/pecotmr_integration/univariate_rss.R \
        --ld-meta-data ${ld_meta_data} \
        --studies "${",".join(regional_data["GWAS"].keys())}" \
        --sumstat-paths "${",".join([regional_data["GWAS"][x][regional_data["regions"][_regions][0]][0] for x in regional_data["GWAS"].keys()])}" \
        --column-file-paths "${",".join([str(regional_data["GWAS"][x][regional_data["regions"][_regions][0]][1]) for x in regional_data["GWAS"].keys()])}" \
        --n-samples "${",".join([str(regional_data["GWAS"][x][regional_data["regions"][_regions][0]][2]) for x in regional_data["GWAS"].keys()])}" \
        --n-cases "${",".join([str(regional_data["GWAS"][x][regional_data["regions"][_regions][0]][3]) for x in regional_data["GWAS"].keys()])}" \
        --n-controls "${",".join([str(regional_data["GWAS"][x][regional_data["regions"][_regions][0]][4]) for x in regional_data["GWAS"].keys()])}" \
        --ld-meta-data-paths "${",".join([str(regional_data["GWAS"][x][regional_data["regions"][_regions][0]][5]) for x in regional_data["GWAS"].keys()])}" \
        --region "chr${_regions.replace(":", "_")}" \
        ${"--skip-regions " + ",".join(skip_regions) if skip_regions else ""} \
        ${"--extract-region-name " + extract_region_name if extract_region_name != "NULL" else ""} \
        ${"--region-name-col " + region_name_col if region_name_col != "NULL" else ""} \
        --imiss ${imiss} \
        --maf ${maf} \
        --L ${L} \
        --L-greedy ${L_greedy} \
        --pip-cutoff ${pip_cutoff} \
        --skip-analysis-pip-cutoff ${skip_analysis_pip_cutoff} \
        --coverage "${",".join([str(x) for x in coverage])}" \
        --ld-reference-size ${ld_reference_size} \
        ${"--ld-mismatch-correction" if ld_mismatch_correction else ""} \
        ${"--finemapping-method " + finemapping_method if finemapping_method else ""} \
        ${"--impute" if impute else ""} \
        --rcond ${rcond} \
        --lamb ${lamb} \
        --r2-threshold ${R2_threshold} \
        --minimum-ld ${minimum_ld} \
        ${"--qc-method " + qc_method if qc_method else ""} \
        ${"--comment-string " + comment_string if comment_string != "NULL" else ""} \
        ${"--diagnostics" if diagnostics else ""} \
        --output-prefix ${_output:nn} \
        --output ${_output}


In [ ]:
[univariate_plot]
parameter: input = path
input: input
output: pip_plot = f"{cwd}/{_input:bn}.png"
task: trunk_workers = 1, trunk_size = job_size, walltime = '12h', mem = '20G', cores = numThreads, tags = f'{step_name}_{_output:bn}'
bash: expand= "${ }", stderr = f'{_output[0]:n}.stderr', stdout = f'{_output[0]:n}.stdout', container = container, entrypoint = entrypoint
    Rscript ${modular_script_dir}/pecotmr_integration/univariate_plot.R \
        --input "${_input}" \
        --output "${_output[0]}"


## Troubleshooting

| Error | Cause | Solution |
|-------|-------|----------|
| `'arg' should be one of "slalom", "dentist"` | An unsupported value was passed to `--qc-method`. | Use `--qc-method slalom` or `--qc-method dentist` (or omit it to skip QC). |
| `ERROR: Incorrect workflow name` | The workflow name was omitted, so SoS read the first `--param` value as a workflow. | Always name the workflow explicitly, e.g. `sos run pipeline/rss_analysis.ipynb univariate_rss --...`. |
| `Failed to locate pipeline/...ipynb.sos` | The `pipeline/` symlink is missing or points to the wrong relative target. | Ensure `pipeline/rss_analysis.ipynb` -> `../code/mnm_analysis/mnm_methods/rss_analysis.ipynb`. |
| `Remaining variants after filter: 0` / empty result | The region has too few well-matched variants after MAF / LD-score / R2 filtering (common for small toy cohorts). | Relax `--maf`, `--minimum-ld`, or `--R2-threshold`, choose a denser region, or supply a larger LD reference. |
| No summary statistics found for region | The region name does not overlap any rows in the summary-statistics file, or the LD-block path is wrong. | Confirm `--region-name chr:start-end` overlaps the data and that the LD-reference `path` resolves relative to the LD-metadata file. |

## Anticipated Results

Produces `.susie_rss.rds` files with SuSiE-RSS fine-mapping posteriors for each GWAS locus against the LD reference.